# IRT Python Library — Usage Examples

This notebook demonstrates how to fit Rasch and 2PL models using the library, compare estimation methods (MML-EM vs JMLE), and score abilities.

In [1]:
import numpy as np

from irt import fit
from irt.core import prob_2pl

np.random.seed(42)

## Simulate response data

We'll simulate dichotomous responses from a 2PL model so we can fit both Rasch and 2PL models to the same data.

In [2]:
N, J = 500, 12

# True parameters
theta_true = np.random.standard_normal(N)
a_true = np.exp(np.random.normal(0.1, 0.3, J))
b_true = np.random.normal(0.0, 1.0, J)

# Generate responses
p = prob_2pl(theta_true[:, None], a_true, b_true)
X = (np.random.rand(N, J) < p).astype(float)

# Inject a small amount of missingness
missing_mask = np.random.rand(N, J) < 0.03
X[missing_mask] = np.nan

X.shape

(500, 12)

## Fit Rasch model (MML-EM)

MML-EM estimates item parameters while integrating over the latent ability distribution using quadrature.

In [3]:
rasch_mml = fit(X, model="rasch", estimator="mml_em")

rasch_mml

FitResult(model='rasch', estimator='mml_em', n_items=12, n_persons=500, n_iter=12, converged, loglik=-3455.46)

## Fit 2PL model (MML-EM)

2PL estimates both discrimination (`a`) and difficulty (`b`) parameters.

In [4]:
twopl_mml = fit(X, model="2pl", estimator="mml_em")

twopl_mml

FitResult(model='2pl', estimator='mml_em', n_items=12, n_persons=500, n_iter=16, converged, loglik=-3444.51)

## Fit Rasch model (JMLE)

JMLE estimates both person abilities and item difficulties jointly (no marginal log-likelihood is computed).

In [5]:
rasch_jmle = fit(X, model="rasch", estimator="jmle")

rasch_jmle

/Users/paulius.satkus/Documents/Item Response Theory Python/irt/api.py:224: RuntimeWarning: 14 persons have extreme scores. Using Bayesian prior for regularization.
  result = fit_jmle(


FitResult(model='rasch', estimator='jmle', n_items=12, n_persons=500, n_iter=6, converged, loglik=-2905.20)

## Estimation method differences (MML-EM vs JMLE)

- **Likelihood**: MML-EM maximizes the *marginal* likelihood by integrating over abilities; JMLE maximizes the *joint* likelihood of persons and items.
- **Identification**: MML-EM uses a prior on ability (default N(0,1)); JMLE centers item difficulties to mean 0.
- **Outputs**: MML-EM provides a marginal log-likelihood and posterior weights; JMLE does not compute marginal log-likelihood.
- **Models supported**: MML-EM supports Rasch and 2PL; JMLE supports Rasch only.
- **Bias/efficiency**: JMLE can be biased in small samples; MML-EM is generally preferred for inference.

In [6]:
# Compare Rasch difficulties from MML-EM vs JMLE (centered)
b_mml = rasch_mml.params["b"]
b_jmle = rasch_jmle.params["b"]

b_mml_centered = b_mml - b_mml.mean()
b_jmle_centered = b_jmle - b_jmle.mean()

corr = np.corrcoef(b_mml_centered, b_jmle_centered)[0, 1]

{
    "corr_centered_b": corr,
    "mml_b_mean": b_mml.mean(),
    "jmle_b_mean": b_jmle.mean(),
}

{'corr_centered_b': np.float64(0.999996953845618),
 'mml_b_mean': np.float64(-0.18812484515813313),
 'jmle_b_mean': np.float64(9.25185853854297e-18)}

## Scoring abilities

- For **MML-EM** fits, you can compute `eap`, `map`, or `mle` scores.
- For **JMLE** fits, `eap` is not available (no posterior), but `map` and `mle` are supported.

In [7]:
scores_eap = rasch_mml.score(method="eap")
scores_map = rasch_mml.score(method="map")
scores_mle = rasch_mml.score(method="mle")

{
    "eap_mean": scores_eap.theta.mean(),
    "map_mean": scores_map.theta.mean(),
    "mle_mean": scores_mle.theta.mean(),
}

{'eap_mean': np.float64(6.162732768168234e-05),
 'map_mean': np.float64(-0.00320000000000004),
 'mle_mean': np.float64(0.06603245744346371)}

In [8]:
jmle_map = rasch_jmle.score(method="map")
jmle_mle = rasch_jmle.score(method="mle")

{
    "jmle_map_mean": jmle_map.theta.mean(),
    "jmle_mle_mean": jmle_mle.theta.mean(),
}

{'jmle_map_mean': np.float64(0.20920252198365438),
 'jmle_mle_mean': np.float64(0.20920252198365438)}

## Custom technical settings

You can override algorithm settings via the `technical` argument.

In [9]:
rasch_mml_fine = fit(
    X,
    model="rasch",
    estimator="mml_em",
    technical={
        "quadpts": 81,
        "max_iter": 300,
        "tol": 1e-5,
    },
)

rasch_mml_fine

FitResult(model='rasch', estimator='mml_em', n_items=12, n_persons=500, n_iter=18, converged, loglik=-3455.46)